# 12. #04 파라미터 + 시드 10개

In [1]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import os

RANDOM_STATE = 42
SEEDS = list(range(42, 52))


## 1. Data Load

In [2]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
sample_submission = pd.read_csv('../data/sample_submission.csv')

train = train.drop_duplicates(subset=[col for col in train.columns if col != 'ID']).reset_index(drop=True)
train['mean_working'] = train['mean_working'].fillna(0)
test['mean_working'] = test['mean_working'].fillna(0)

for col in ['medical_history', 'family_medical_history']:
    train[col] = train[col].fillna('None')
    test[col] = test[col].fillna('None')

train['edu_level'] = train['edu_level'].fillna('Unknown')
test['edu_level'] = test['edu_level'].fillna('Unknown')

print('train:', train.shape, '/ test:', test.shape)


train: (2994, 18) / test: (3000, 17)


## 2. 파생변수 생성

In [3]:
def add_features(df):
    data = df.copy()
    has_disease = (data['medical_history'] != 'None').astype(int)

    data['is_overworking'] = (data['mean_working'] >= 10).astype(int)
    data['work_sleep_risk'] = ((data['mean_working'] >= 9) & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['oversleep_low_activity'] = ((data['sleep_pattern'] == 'oversleeping') & (data['activity'] == 'light')).astype(int)
    data['working_age_ratio'] = data['mean_working'] / (data['age'] + 1)
    data['activity_sleep_mismatch'] = ((data['activity'] == 'intense') & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)

    data['smoker_with_disease'] = ((data['smoke_status'] == 'current-smoker') & (has_disease == 1)).astype(int)
    data['age_disease_interaction'] = data['age'] * has_disease
    data['has_medical_history'] = has_disease
    data['has_family_history'] = (data['family_medical_history'] != 'None').astype(int)
    data['total_disease_burden'] = data['has_medical_history'] + data['has_family_history']
    data['genetic_risk_match'] = ((data['medical_history'] == data['family_medical_history']) & (has_disease == 1)).astype(int)
    data['anticipatory_stress'] = ((data['family_medical_history'] != 'None') & (data['medical_history'] == 'None')).astype(int)

    data['bmi'] = data['weight'] / ((data['height'] / 100) ** 2)
    data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']
    data['map'] = data['diastolic_blood_pressure'] + (data['pulse_pressure'] / 3)
    data['is_hypertension'] = ((data['systolic_blood_pressure'] >= 140) | (data['diastolic_blood_pressure'] >= 90)).astype(int)
    data['cardio_metabolic_load'] = data['map'] * data['bmi']

    data['is_low_bone_density'] = (data['bone_density'] < 0).astype(int)
    data['glucose_chol_ratio'] = data['glucose'] / (data['cholesterol'] + 1)

    return data


train = add_features(train)
test = add_features(test)
print('파생변수 적용 후 train shape:', train.shape)


파생변수 적용 후 train shape: (2994, 37)


## 3. 인코딩

In [4]:
activity_map = {'light': 0, 'moderate': 1, 'intense': 2}
edu_map = {'Unknown': 0, 'high school diploma': 1, 'bachelors degree': 2, 'graduate degree': 3}

train['activity'] = train['activity'].map(activity_map)
test['activity'] = test['activity'].map(activity_map)
train['edu_level'] = train['edu_level'].map(edu_map)
test['edu_level'] = test['edu_level'].map(edu_map)

nominal_cols = ['gender', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern']

for feature in nominal_cols:
    le = LabelEncoder()
    le = le.fit(train[feature])
    train[feature] = le.transform(train[feature])

    unseen = [label for label in np.unique(test[feature]) if label not in le.classes_]
    if unseen:
        le.classes_ = np.append(le.classes_, unseen)
    test[feature] = le.transform(test[feature])

x_train = train.drop(['ID', 'stress_score'], axis=1)
y_train = train['stress_score']
x_test = test.drop('ID', axis=1)

print('x_train:', x_train.shape, '/ x_test:', x_test.shape)


x_train: (2994, 35) / x_test: (3000, 35)


## 4. 시드별 5-Fold CV

In [5]:
best_params_04 = dict(
    n_estimators=1200,
    learning_rate=0.08,
    reg_alpha=0.15,
    reg_lambda=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
)

seed_oof_list = []
seed_test_list = []

for seed in SEEDS:
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    seed_oof = np.zeros(len(x_train))
    seed_test = np.zeros(len(x_test))

    for tr_idx, va_idx in kf.split(x_train):
        X_tr, X_va = x_train.iloc[tr_idx], x_train.iloc[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

        model = LGBMRegressor(**best_params_04, random_state=seed, verbose=-1)
        model.fit(X_tr, y_tr)

        seed_oof[va_idx] = model.predict(X_va)
        seed_test += model.predict(x_test) / kf.n_splits

    seed_oof_list.append(seed_oof)
    seed_test_list.append(seed_test)
    print(f'seed={seed} MAE = {mean_absolute_error(y_train, seed_oof):.4f}')


seed=42 MAE = 0.1815
seed=43 MAE = 0.1818
seed=44 MAE = 0.1841
seed=45 MAE = 0.1846
seed=46 MAE = 0.1795
seed=47 MAE = 0.1810
seed=48 MAE = 0.1825
seed=49 MAE = 0.1801
seed=50 MAE = 0.1834
seed=51 MAE = 0.1827


In [6]:
oof_matrix = np.column_stack(seed_oof_list)
test_matrix = np.column_stack(seed_test_list)

for k in [1, 3, 5, 7, 10]:
    if k > len(SEEDS):
        continue
    cum_mae = mean_absolute_error(y_train, oof_matrix[:, :k].mean(axis=1))
    print(f'시드 {k:2d}개 평균 MAE = {cum_mae:.4f}')

full_mae = mean_absolute_error(y_train, oof_matrix.mean(axis=1))
print(f'\n10-seed 평균 MAE = {full_mae:.4f}')


시드  1개 평균 MAE = 0.1815
시드  3개 평균 MAE = 0.1786
시드  5개 평균 MAE = 0.1778
시드  7개 평균 MAE = 0.1771
시드 10개 평균 MAE = 0.1769

10-seed 평균 MAE = 0.1769


## 5. 제출 파일 저장

In [7]:
os.makedirs('../submissions', exist_ok=True)
final_pred_12 = np.clip(test_matrix.mean(axis=1), 0, 1)
sample_submission['stress_score'] = final_pred_12
sample_submission.to_csv('../submissions/submit_12_multiseed_04_10seeds.csv', index=False)
sample_submission.head()


,ID,stress_score
0,TEST_0000,0.575057
1,TEST_0001,0.861482
2,TEST_0002,0.263784
3,TEST_0003,0.490644
4,TEST_0004,0.589811
